In [1]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
# import threading
mt5.initialize()


True

In [2]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("EURUSD", 0.02, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

38.78

In [17]:
def get_values(symbol):
    timezone = pytz.timezone("Etc/UTC")
    x = datetime.now()
    utc_from = datetime(2021, 4, 1, tzinfo=timezone)
    utc_to = datetime(x.year, x.month+1, x.day, tzinfo=timezone)
    # utc_to = datetime(x.year, x.month+1, 1, tzinfo=timezone)
    rates = mt5.copy_rates_range(symbol, mt5.TIMEFRAME_H1, utc_from, utc_to)

    rates_frame = pd.DataFrame(rates)
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume'], axis=1)
    # convert time in seconds into the datetime format
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
#     rates_frame['rsi'] = pta.rsi(rates_frame['close'], length = 14)
    
#     rates_frame['sma'] = rates_frame['close'].rolling(window=100).mean()
#     rates_frame['smaH'] = rates_frame['high'].rolling(window=20).mean()
#     rates_frame['smaL']= rates_frame['low'].rolling(window=20).mean()
    rates_frame = rates_frame.drop(['high', 'low'], axis=1)
    return rates_frame

In [23]:
def get_rsi(close, lookback):
    ret = close.diff()
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
    rsi_df = rsi_df.dropna()
    return rsi_df[3:]

# ibm['rsi_14'] = get_rsi(ibm['close'], 14)
# ibm = ibm.dropna()


In [75]:
symbol = "GBPUSD"
a= get_values(symbol)
a['rsi'] = get_rsi(a['close'], 14)
a = a.dropna()

In [82]:
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
mul = 100000

for i in range(7, len(a)):
    if str(a.iloc[i].rsi) != 'nan':
        if round(a.iloc[i].rsi, 2) > 62.0 and int(round(a.iloc[i-1].rsi, 2)*100) in range(int(59.0*100), int(63.5*100)) \
        and a.iloc[i].rsi > a.iloc[i-1].rsi and a.iloc[i-2].rsi > a.iloc[i-1].rsi \
        and a.iloc[i-2].rsi > 67.0 and check == 0 :
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print(round(a.iloc[i].rsi,2))
            
            print("*"*20)
            check = 1  

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(pp,"---", round(a.iloc[i].rsi, 2))
            
            if pp < -3.0:
                profit.append(pp)
                check = 0
            elif round(a.iloc[i].rsi, 2) <= 59.0:
                profit.append(pp)
                check = 0

####################
2021-04-05 13:00:00
66.16
********************
-0.08 --- 65.87
2.56 --- 70.51
6.32 --- 75.6
6.0 --- 74.42
8.5 --- 77.39
7.72 --- 74.49
8.08 --- 74.95
8.0 --- 74.63
8.16 --- 74.86
7.04 --- 69.96
8.44 --- 72.39
9.64 --- 74.31
9.72 --- 74.44
7.86 --- 66.2
7.3 --- 63.91
5.9 --- 58.46
####################
2021-04-14 11:00:00
65.35
********************
-4.92 --- 53.73
####################
2021-04-19 01:00:00
63.07
********************
-2.7199999999999998 --- 55.99
####################
2021-05-07 17:00:00
70.91
********************
4.16 --- 74.4
0.58 --- 66.96
2.24 --- 68.53
3.54 --- 69.75
4.66 --- 70.79
1.56 --- 64.18
3.74 --- 66.55
10.06 --- 72.27
12.28 --- 73.95
9.84 --- 68.99
10.52 --- 69.6
7.86 --- 64.26
8.22 --- 64.66
9.28 --- 65.85
12.8 --- 69.54
17.1 --- 73.34
19.36 --- 75.09
22.14 --- 77.09
22.42 --- 77.29
24.2 --- 78.55
26.2 --- 79.91
33.62 --- 83.96
26.38 --- 69.28
32.12 --- 73.27
34.46 --- 74.71
32.68 --- 71.55
32.96 --- 71.75
31.08 --- 68.24
30.62 --- 67.37
2

In [83]:
sum(profit)

27.800000000000004

In [84]:
n = 0
p = 0
tn = 0
tp = 0
for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}")      

Total negative sm -->-15.079999999999998
Total negative -->6
Total positive sm -->42.88
Total positive -->4


In [ ]:
counterr = 0
peck = 0
for i in profit:
    if i < 0.0:
        counterr = counterr + i
    else:
        peck = peck + i
print(counterr)
print(peck)

In [ ]:
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
mul = 100000

for i in range(7, len(a)):
    if str(a.iloc[i].sma) != 'nan':
        
        if a.iloc[i].close > a.iloc[i].smaH and a.iloc[i].open > a.iloc[i].smaH \
            and a.iloc[i-1].close > a.iloc[i-1].smaH and a.iloc[i-1].open > a.iloc[i-1].smaH \
            and up == 0:
            up = 1
        elif up == 1 and a.iloc[i].close < a.iloc[i].smaL and a.iloc[i].open < a.iloc[i].smaL and check == 0:
            #if candle has crossed smaL
            if a.iloc[i].close < a.iloc[i].open \
            and int(a.iloc[i].sma*mul) not in range(int(a.iloc[i].smaL*mul), int(a.iloc[i].smaH*mul)) \
            and a.iloc[i].close < a.iloc[i].sma and a.iloc[i].open < a.iloc[i].sma: 
                #checks if next candle is not in reverse of sell or up candle
                buy_price = a.iloc[i].close
                print("#"*20)
                print(a.iloc[i].name)
                print("*"*20)
                check = 1  
                peck = 0

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
            print(pp,"---",a.iloc[i].name)
#             profit.append(pp)
#             check = 0
#             up = 0
                
#             if pp < -2.0 and peck == 0:
#                 print(a.iloc[i].name)
#                 profit.append(pp)
#                 check = 0
#                 up = 0
#                 peck = 1
            if a.iloc[i].close > a.iloc[i].smaL:
#                 if peck == 1:
                    print(a.iloc[i].name)
                    profit.append(pp)
                    check = 0
                    up = 0
#                 else:
#                     print(a.iloc[i].name)
#                     profit.append(2*pp)
#                     check = 0
#                     up = 0
            
            
#             if 

            
            


In [ ]:
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
mul = 100000

for i in range(7, len(a)):
#     if str(a.iloc[i].sma) != 'nan':
        hd = a.iloc[i].close - a.iloc[i].open
        ld = a.iloc[i].open - a.iloc[i].close
        base = 1.17610 - 1.17600
        
#         if and a.iloc[i].open < a.iloc[i].smaL and a.iloc[i].close < a.iloc[i].smaL \
        if a.iloc[i].open < a.iloc[i].close \
        and a.iloc[i-1].open > a.iloc[i-1].close \
        and a.iloc[i-2].open > a.iloc[i-2].close \
        and check == 0 :
#         and hd > base:           
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print("*"*20)
            check = 1  
            peck = 0

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.05, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(pp,"---",a.iloc[i].name)
            
            check = 0
            if pp < 0.0:
                sell_price = a.iloc[i+1].close
                pp = price_action(symbol, 0.05, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
                profit.append(pp)
            else:
                profit.append(pp)